# Notebook 02 - Adapt Dataset With Adaption

This notebook follows the Adaption SDK flow from the docs: install/import the SDK, load credentials, upload the prepared file, confirm column mapping, run an estimate, adapt the rows that Adaption accepted during upload, poll for completion, download the adapted output, and inspect before/after rows.

**Files read:**
- [`../configs/adaption_run.yaml`](../configs/adaption_run.yaml) - paths, column mapping, dynamic job-size behavior, and timeout settings.
- [`../configs/adaption_blueprint.yaml`](../configs/adaption_blueprint.yaml) - qualitative instructions and Adaption recipe settings.
- [`../data/processed/kakugo_adaption_input.csv`](../data/processed/kakugo_adaption_input.csv) - prepared prompt/response rows from Notebook 01.

**Files written:**
- [`../data/adapted/kakugo_adapted.csv`](../data/adapted/kakugo_adapted.csv) - adapted dataset downloaded from Adaption.
- [`../data/processed/kakugo_adapted_sft.jsonl`](../data/processed/kakugo_adapted_sft.jsonl) - adapted data converted to chat-format JSONL for SFT.
- [`../results/adaption_run_metadata.json`](../results/adaption_run_metadata.json) - run metadata, including the uploaded dataset ID and run plan.
- [`../results/adaption_dataset_diagnosis.json`](../results/adaption_dataset_diagnosis.json) - API-visible dataset description, status, and evaluation metadata captured before the adaptation run.
- [`../results/adaption_dataset_evaluation.json`](../results/adaption_dataset_evaluation.json) - API-visible quality/evaluation metadata captured after the adaptation run.


In [ ]:
import json
import os
import sys
from pathlib import Path
import pandas as pd

from adaption import Adaption

In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs' / 'project.yaml').exists():
            return candidate
    raise FileNotFoundError(
        'Run this notebook from inside the adaption-kirundi-sft-starter repo.'
    )

ROOT = find_repo_root(Path.cwd())
SRC = str(ROOT / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
ROOT_STR = str(ROOT)
if ROOT_STR not in sys.path:
    sys.path.insert(0, ROOT_STR)

In [ ]:
from kirundi_sft_starter.adaption import (
    blueprint_text,
    capture_dataset_diagnosis,
    download_to_file,
    get_adaptation_job_config,
    to_plain_data,
    wait_for_completion,
    wait_until_ingested,
)
from kirundi_sft_starter.data import convert_adapted_to_sft
from kirundi_sft_starter.utils import ensure_dir, load_env, load_yaml

load_env()

In [ ]:
project_config = load_yaml(ROOT / 'configs/project.yaml')
adaption_run_config = load_yaml(ROOT / 'configs/adaption_run.yaml')
adaptation_job_config = get_adaptation_job_config(adaption_run_config)

blueprint_config = load_yaml(ROOT / 'configs/adaption_blueprint.yaml')

if not os.environ.get('ADAPTION_API_KEY'):
    raise RuntimeError('Missing ADAPTION_API_KEY. Add it to .env before running this notebook.')

## Inspect the input and column mapping

Adaption needs to know which columns contain the prompt and completion. This repo maps `instruction` to `prompt` and `response` to `completion`.

In [ ]:
adaption_input_path = ROOT / adaption_run_config['input_path']

adaption_input_df = pd.read_csv(adaption_input_path)

print(f"Size: {adaption_input_df.shape[0]}")

adaption_input_df.head()

In [ ]:
adaption_run_config['column_mapping']

In [ ]:
print(f"Path: ROOT/{'configs/adaption_blueprint.yaml'}\n")
      
brand_controls = dict(blueprint_config['adaption_brand_controls'])

brand_controls['blueprint'] = blueprint_text(blueprint_config)

print(brand_controls['blueprint'][:1000])

## Upload the dataset



In [ ]:
client = Adaption(api_key=os.environ['ADAPTION_API_KEY'])
upload = client.datasets.upload_file(str(adaption_input_path), name=adaption_run_config['dataset_name'])
dataset_id = upload.dataset_id
print('Dataset ID:', dataset_id)

ingested_status = wait_until_ingested(
    client,
    dataset_id,
    timeout_seconds=adaption_run_config.get('ingestion_timeout_seconds'),
)
ingested_status_data = to_plain_data(ingested_status)
print('Dataset ingestion metadata:', ingested_status_data)

api_row_count = ingested_status_data.get('row_count')
if api_row_count is None:
    raise RuntimeError('Adaption did not report a row_count for the uploaded dataset.')

accepted_rows = int(api_row_count)
if accepted_rows <= 0:
    raise RuntimeError(f'Adaption reported row_count={accepted_rows}; cannot launch an adaptation run.')

configured_max_rows = adaptation_job_config.get('max_rows')
if configured_max_rows is None:
    requested_rows = accepted_rows
else:
    requested_rows = min(int(configured_max_rows), accepted_rows)

adaptation_job_config = dict(adaptation_job_config)
adaptation_job_config['max_rows'] = requested_rows

print(
    f"Local CSV rows: {len(adaption_input_df)} | "
    f"Adaption accepted rows: {accepted_rows} | "
    f"Rows requested for adaptation: {requested_rows}"
)

## Capture API-visible diagnosis metadata

The Adaption UI may show a richer **Data Diagnosis** page before launch. The public API docs expose the closest quick metadata through dataset get/status/list calls. This cell intentionally does not call `get_evaluation` before the adaptation run because that endpoint can block before evaluation output exists.

In [ ]:
diagnosis_path = ROOT / adaption_run_config.get(
    'diagnosis_path',
    'results/adaption_dataset_diagnosis.json',
)
ensure_dir(diagnosis_path)

diagnosis_metadata = capture_dataset_diagnosis(
    client,
    dataset_id,
    adaption_run_config['dataset_name'],
)
diagnosis_path.write_text(
    json.dumps(diagnosis_metadata, indent=2, ensure_ascii=False, default=str),
    encoding='utf-8',
)

dataset_record = diagnosis_metadata.get('dataset_record') or {}
dataset_status = diagnosis_metadata.get('dataset_status') or {}
listed_dataset = diagnosis_metadata.get('listed_dataset') or {}

summary = {
    'dataset_id': dataset_id,
    'name': listed_dataset.get('name') or dataset_record.get('name'),
    'description': listed_dataset.get('description') or dataset_record.get('description'),
    'status': dataset_status.get('status') or listed_dataset.get('status'),
    'row_count': dataset_status.get('row_count') or listed_dataset.get('row_count'),
    'evaluation': 'retrieved after adaptation run',
    'saved_to': str(diagnosis_path.relative_to(ROOT)),
}
pd.DataFrame([summary]).T.rename(columns={0: 'value'})

## Run estimate

This cell asks Adaption for an estimate before launching the actual adaptation run. The estimate uses the row count that Adaption accepted during upload.

In [ ]:
estimate = client.datasets.run(
    dataset_id,
    column_mapping=adaption_run_config['column_mapping'],
    brand_controls=brand_controls,
    recipe_specification=blueprint_config['adaption_recipe_specification'],
    job_specification={'max_rows': adaptation_job_config['max_rows']},
    estimate=True,
)
print('Estimated credits:', estimate.estimated_credits_consumed)

## Run adaptation job and save outputs

This starts the configured adaptation job, waits for completion, downloads the adapted CSV, and writes run metadata locally. The requested row count comes from Adaption's accepted upload row count unless `configs/adaption_run.yaml` sets an explicit smaller cap. In `configs/adaption_run.yaml`, `timeout_seconds: null` means keep waiting instead of failing after a fixed time. The wait cell prints a heartbeat every 60 seconds.

In [ ]:
adaption_job = client.datasets.run(
    dataset_id,
    column_mapping=adaption_run_config['column_mapping'],
    brand_controls=brand_controls,
    recipe_specification=blueprint_config['adaption_recipe_specification'],
    job_specification={'max_rows': adaptation_job_config['max_rows']},
)
print('Adaption run started:', adaption_job.run_id)

In [ ]:
final = wait_for_completion(
    client,
    dataset_id,
    timeout_seconds=adaptation_job_config['timeout_seconds'],
    heartbeat_seconds=60,
)
print('Adaption run finished:', final.status)
if getattr(final, 'error', None):
    raise RuntimeError(final.error.message)

## Retrieve Adaption dataset and evaluation metadata


In [ ]:
evaluation_path = ROOT / adaption_run_config.get(
    'evaluation_path',
    'results/adaption_dataset_evaluation.json',
)
ensure_dir(evaluation_path)

post_run_evaluation = client.datasets.get_evaluation(dataset_id)
post_run_evaluation_data = to_plain_data(post_run_evaluation)
evaluation_path.write_text(
    json.dumps(post_run_evaluation_data, indent=2, ensure_ascii=False, default=str),
    encoding='utf-8',
)

def find_nested_value(value, key):
    if isinstance(value, dict):
        if key in value:
            return value[key]
        for item in value.values():
            found = find_nested_value(item, key)
            if found is not None:
                return found
    elif isinstance(value, list):
        for item in value:
            found = find_nested_value(item, key)
            if found is not None:
                return found
    return None

In [ ]:
evaluation_summary = {
    'grade_before': find_nested_value(post_run_evaluation_data, 'grade_before'),
    'grade_after': find_nested_value(post_run_evaluation_data, 'grade_after'),
    'score_before': find_nested_value(post_run_evaluation_data, 'score_before'),
    'score_after': find_nested_value(post_run_evaluation_data, 'score_after'),
    'status': find_nested_value(post_run_evaluation_data, 'status'),
    'saved_to': str(evaluation_path.relative_to(ROOT)),
}
pd.DataFrame([evaluation_summary]).T.rename(columns={0: 'value'})

In [ ]:
adapted_output_path = ROOT / adaption_run_config['output_path']
metadata_path = ROOT / adaption_run_config['metadata_path']

download_to_file(client, dataset_id, adapted_output_path)

ensure_dir(metadata_path)
metadata = {
    'dataset_id': dataset_id,
    'adaption_run_id': adaption_job.run_id,
    'input_path': str(adaption_input_path.relative_to(ROOT)),
    'output_path': str(adapted_output_path.relative_to(ROOT)),
    'diagnosis_path': str(diagnosis_path.relative_to(ROOT)),
    'evaluation_path': str(evaluation_path.relative_to(ROOT)) if 'evaluation_path' in globals() else None,
    'column_mapping': adaption_run_config['column_mapping'],
    'adaptation_job': adaptation_job_config,
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print('Adapted CSV:', adapted_output_path)
print('Run metadata:', metadata_path)

In [ ]:
adapted_df = pd.read_csv(adapted_output_path)
print('Adapted rows:', len(adapted_df))
adapted_df.head()

In [ ]:
metadata_path = ROOT / adaption_run_config['metadata_path']

with metadata_path.open('r', encoding='utf-8') as f:
    adaption_run_metadata = json.load(f)

pd.DataFrame([adaption_run_metadata]).T.rename(columns={0: 'value'})

## Convert for SFT

Convert the adapted CSV into the same chat-format JSONL structure used by the raw SFT run.

In [ ]:
adapted_sft_path = ROOT / project_config['datasets']['sft']['adapted_sft_path']
adapted_sft_df = convert_adapted_to_sft(adapted_output_path, adapted_sft_path)

print('Converted adapted examples:', len(adapted_sft_df))
print('Adapted SFT JSONL:', adapted_sft_path)

In [ ]:
with adapted_sft_path.open('r', encoding='utf-8') as f:
    adapted_sft_preview_rows = [json.loads(line) for _, line in zip(range(3), f)]

pd.DataFrame(
    [
        {
            'example_id': row.get('metadata', {}).get('example_id'),
            'user': row['messages'][0]['content'],
            'assistant': row['messages'][1]['content'],
        }
        for row in adapted_sft_preview_rows
    ]
)